# 02. Baselines & Ridge Regression
Triển khai đối chuẩn:
- `Persistence Baseline`: $\hat{y}_{t+h} = 	ext{lag}_0$.
- `Climatology Baseline`: Trung bình lịch sử theo `[station_id, month, hour]` trên Train.
- `Ridge Regressor`: Điều chuẩn L2, `StandardScaler` (chỉ fit trên Train).


In [1]:
import os
import time
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

BASE_DIR = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
if not os.path.exists(PROCESSED_DIR):
    PROCESSED_DIR = os.path.join(BASE_DIR, "data_processed")
ARTIFACTS_DIR = os.path.join(BASE_DIR, "artifacts")
os.makedirs(f"{ARTIFACTS_DIR}/predictions", exist_ok=True)
os.makedirs(f"{ARTIFACTS_DIR}/metrics", exist_ok=True)
os.makedirs(f"{ARTIFACTS_DIR}/models", exist_ok=True)

def load_data(scenario):
    p_train = os.path.join(PROCESSED_DIR, f"dataset_{scenario}_train.parquet")
    p_val = os.path.join(PROCESSED_DIR, f"dataset_{scenario}_val.parquet")
    p_test = os.path.join(PROCESSED_DIR, f"dataset_{scenario}_test.parquet")
    if os.path.exists(p_train):
        return pd.read_parquet(p_train), pd.read_parquet(p_val), pd.read_parquet(p_test)
    return pd.read_csv(p_train.replace(".parquet", ".csv")), pd.read_csv(p_val.replace(".parquet", ".csv")), pd.read_csv(p_test.replace(".parquet", ".csv"))

def calculate_metrics(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    smape = np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8)) * 100
    
    act_150 = (y_true > 150)
    pred_150 = (y_pred > 150)
    tp_150 = np.sum(act_150 & pred_150)
    recall_150 = tp_150 / (np.sum(act_150) + 1e-8)
    prec_150 = tp_150 / (np.sum(pred_150) + 1e-8)
    f1_150 = 2 * prec_150 * recall_150 / (prec_150 + recall_150 + 1e-8)
    return {
        "R2": float(r2), "MAE": float(mae), "RMSE": float(rmse),
        "sMAPE": float(smape), "Recall_150": float(recall_150),
        "Precision_150": float(prec_150), "F1_150": float(f1_150)
    }


### Huấn Luyện & Đánh Giá


In [2]:
all_results = []

for sc in ["1h", "24h"]:
    print(f"--- Kịch bản: {sc.upper()} ---")
    train_df, val_df, test_df = load_data(sc)
    pred_df = test_df[['time', 'station_id', 'target']].copy().rename(columns={'target': 'actual'})
    
    # 1. Persistence
    t0 = time.time()
    val_pred_pers = val_df['target_aqi_lag_0'].values
    test_pred_pers = test_df['target_aqi_lag_0'].values
    time_pers = time.time() - t0
    
    m_val = calculate_metrics(val_df['target'], val_pred_pers)
    m_test = calculate_metrics(test_df['target'], test_pred_pers)
    pred_df['pred_persistence'] = test_pred_pers
    all_results.append({
        "scenario": sc, "family": "Baseline", "model": "Persistence",
        "val_R2_2024": m_val["R2"], "val_MAE_2024": m_val["MAE"],
        "test_R2_2025": m_test["R2"], "test_MAE_2025": m_test["MAE"],
        "test_RMSE_2025": m_test["RMSE"], "test_sMAPE_2025": m_test["sMAPE"],
        "test_F1_150": m_test["F1_150"], "time_sec": time_pers
    })
    
    # 2. Climatology
    t0 = time.time()
    for d in [train_df, val_df, test_df]:
        d['m'] = pd.to_datetime(d['time']).dt.month
        d['h'] = pd.to_datetime(d['time']).dt.hour
    clim_map = train_df.groupby(['station_id', 'm', 'h'])['target'].mean().to_dict()
    g_mean = train_df['target'].mean()
    
    val_pred_clim = [clim_map.get((r['station_id'], r['m'], r['h']), g_mean) for _, r in val_df.iterrows()]
    test_pred_clim = [clim_map.get((r['station_id'], r['m'], r['h']), g_mean) for _, r in test_df.iterrows()]
    time_clim = time.time() - t0
    
    m_val = calculate_metrics(val_df['target'], val_pred_clim)
    m_test = calculate_metrics(test_df['target'], test_pred_clim)
    pred_df['pred_climatology'] = test_pred_clim
    all_results.append({
        "scenario": sc, "family": "Baseline", "model": "Climatology",
        "val_R2_2024": m_val["R2"], "val_MAE_2024": m_val["MAE"],
        "test_R2_2025": m_test["R2"], "test_MAE_2025": m_test["MAE"],
        "test_RMSE_2025": m_test["RMSE"], "test_sMAPE_2025": m_test["sMAPE"],
        "test_F1_150": m_test["F1_150"], "time_sec": time_clim
    })
    
    # 3. Ridge Regression
    t0 = time.time()
    drop_cols = ['time', 'year', 'target', 'm', 'h']
    feat_cols = [c for c in train_df.columns if c not in drop_cols]
    
    X_train = pd.get_dummies(train_df[feat_cols], columns=['station_id'], drop_first=True)
    X_val = pd.get_dummies(val_df[feat_cols], columns=['station_id'], drop_first=True).reindex(columns=X_train.columns, fill_value=0)
    X_test = pd.get_dummies(test_df[feat_cols], columns=['station_id'], drop_first=True).reindex(columns=X_train.columns, fill_value=0)
    
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test)
    
    ridge = Ridge(alpha=10.0, random_state=42)
    ridge.fit(X_train_s, train_df['target'])
    val_pred_ridge = ridge.predict(X_val_s)
    test_pred_ridge = ridge.predict(X_test_s)
    time_ridge = time.time() - t0
    
    m_val = calculate_metrics(val_df['target'], val_pred_ridge)
    m_test = calculate_metrics(test_df['target'], test_pred_ridge)
    pred_df['pred_ridge'] = test_pred_ridge
    
    joblib.dump(ridge, f"{ARTIFACTS_DIR}/models/model_ridge_{sc}.joblib")
    joblib.dump(scaler, f"{ARTIFACTS_DIR}/models/scaler_ridge_{sc}.joblib")
    
    all_results.append({
        "scenario": sc, "family": "Linear", "model": "Ridge Regression",
        "val_R2_2024": m_val["R2"], "val_MAE_2024": m_val["MAE"],
        "test_R2_2025": m_test["R2"], "test_MAE_2025": m_test["MAE"],
        "test_RMSE_2025": m_test["RMSE"], "test_sMAPE_2025": m_test["sMAPE"],
        "test_F1_150": m_test["F1_150"], "time_sec": time_ridge
    })
    
    pred_df.to_csv(f"{ARTIFACTS_DIR}/predictions/preds_{sc}_baselines_linear.csv", index=False)

df_res = pd.DataFrame(all_results)
df_res.to_json(f"{ARTIFACTS_DIR}/metrics/metrics_baselines_linear.json", orient="records", indent=2)
df_res[['scenario', 'model', 'val_R2_2024', 'test_R2_2025', 'test_MAE_2025', 'test_RMSE_2025', 'time_sec']]


--- Kịch bản: 1H ---


--- Kịch bản: 24H ---


,scenario,model,val_R2_2024,test_R2_2025,test_MAE_2025,test_RMSE_2025,time_sec
0,1h,Persistence,0.886232,0.948194,3.608138,6.027016,0.000000
1,1h,Climatology,0.188806,0.149730,17.532101,24.416999,2.055316
2,1h,Ridge Regression,0.893137,0.947777,3.723427,6.051219,0.231390
3,24h,Persistence,0.277316,0.459721,13.023096,19.414532,0.000000
4,24h,Climatology,0.195270,0.155154,17.445873,24.277662,2.368855
5,24h,Ridge Regression,0.498700,0.585913,11.862214,16.996676,0.235945
